# 01 · SQL data prep

**Goal:** load the raw Hillstrom CSV into DuckDB, clean it with saved SQL queries, and check data quality before any analysis.

All queries live in [`sql/`](../sql) and are run from Python with `hillstrom.data.run_sql_file`. The raw CSV in `data/raw/` is never modified; every cleaning step happens in SQL.

Why DuckDB: it reads the CSV directly with no database server to install, runs analytical SQL fast, and returns results as pandas DataFrames. Anyone who clones the repo can re-run everything with `pip install`.

In [1]:
import pandas as pd
from IPython.display import Markdown, display

from hillstrom.data import connect, run_sql_file
from hillstrom.paths import RAW_CSV, SQL_DIR

pd.set_option("display.float_format", "{:,.4f}".format)


def show_sql(name: str) -> None:
    # Render a saved query in the notebook so the SQL is visible next to its result.
    display(Markdown(f"**`sql/{name}`**\n```sql\n{(SQL_DIR / name).read_text(encoding='utf-8')}\n```"))


con = connect(RAW_CSV)  # in-memory DuckDB; runs sql/01_load_clean.sql

## 1. Load and clean

The load query fixes the column types explicitly (no type guessing), adds a `customer_id` equal to the row position in the raw file, and adds two derived columns:

- `arm`: short labels for the three experiment groups (`mens`, `womens`, `control`).
- `purchase_group`: whether the customer bought men's merchandise, women's, or both in the past year. This is a **pre-treatment** variable, so it is safe to use for segment analysis later.

It also fixes a spelling quirk in the raw file: `zip_code` contains `Surburban`, recoded to `Suburban`.

In [2]:
show_sql("01_load_clean.sql")

**`sql/01_load_clean.sql`**
```sql
-- Load the raw Hillstrom CSV into a typed, cleaned `customers` table.
-- Parameter: $csv_path (path to the raw CSV). The raw file itself is never modified.
--
-- Cleaning steps:
--   * explicit column types instead of type sniffing
--   * customer_id = row position in the raw file (1-based)
--   * zip_code: recode the raw spelling 'Surburban' to 'Suburban'
--   * arm: short treatment labels used throughout the analysis
--   * purchase_group: past-year purchase history (mens only / womens only / both),
--     a pre-treatment variable used for segment analysis

CREATE OR REPLACE TABLE customers AS
WITH raw AS (
    SELECT *
    FROM read_csv(
        $csv_path,
        header = true,
        columns = {
            'recency': 'INTEGER',
            'history_segment': 'VARCHAR',
            'history': 'DOUBLE',
            'mens': 'INTEGER',
            'womens': 'INTEGER',
            'zip_code': 'VARCHAR',
            'newbie': 'INTEGER',
            'channel': 'VARCHAR',
            'segment': 'VARCHAR',
            'visit': 'INTEGER',
            'conversion': 'INTEGER',
            'spend': 'DOUBLE'
        }
    )
)
SELECT
    row_number() OVER () AS customer_id,
    recency,
    history_segment,
    history,
    mens,
    womens,
    CASE
        WHEN mens = 1 AND womens = 1 THEN 'both'
        WHEN mens = 1 THEN 'mens_only'
        WHEN womens = 1 THEN 'womens_only'
    END AS purchase_group,
    CASE WHEN zip_code = 'Surburban' THEN 'Suburban' ELSE zip_code END AS zip_code,
    newbie,
    channel,
    segment,
    CASE segment
        WHEN 'Mens E-Mail' THEN 'mens'
        WHEN 'Womens E-Mail' THEN 'womens'
        WHEN 'No E-Mail' THEN 'control'
    END AS arm,
    visit,
    conversion,
    spend
FROM raw;

```

In [3]:
con.sql("SELECT * FROM customers LIMIT 5").df()

,customer_id,recency,history_segment,history,mens,womens,purchase_group,zip_code,newbie,channel,segment,arm,visit,conversion,spend
0,1,10,2) $100 - $200,142.4400,1,0,mens_only,Suburban,0,Phone,Womens E-Mail,womens,0,0,0.0000
1,2,6,3) $200 - $350,329.0800,1,1,both,Rural,1,Web,No E-Mail,control,0,0,0.0000
2,3,7,2) $100 - $200,180.6500,0,1,womens_only,Suburban,1,Web,Womens E-Mail,womens,0,0,0.0000
3,4,9,5) $500 - $750,675.8300,1,0,mens_only,Rural,1,Web,Mens E-Mail,mens,0,0,0.0000
4,5,2,1) $0 - $100,45.3400,1,0,mens_only,Urban,0,Web,Womens E-Mail,womens,0,0,0.0000


In [4]:
# After cleaning, zip_code has three correctly spelled levels.
con.sql("SELECT zip_code, count(*) AS n FROM customers GROUP BY 1 ORDER BY 1").df()

,zip_code,n
0,Rural,9563
1,Suburban,28776
2,Urban,25661


## 2. Data quality checks

Before estimating anything, check that the data is internally consistent. Each column below counts rows that break a rule and should be zero:

- no missing values, every row mapped to an arm, unique ids, valid amounts;
- outcome logic: spend > 0 exactly when the customer converted, and every converter visited;
- every customer has some past-year purchase history (the population is past-year buyers).

In [5]:
show_sql("04_data_quality.sql")

**`sql/04_data_quality.sql`**
```sql
-- Data quality checks on the cleaned `customers` table (from 01_load_clean.sql).
-- Returns one row. Every column except n_rows counts violating rows and should be 0.

SELECT
    count(*) AS n_rows,
    count(*) FILTER (
        WHERE recency IS NULL OR history_segment IS NULL OR history IS NULL
           OR mens IS NULL OR womens IS NULL OR zip_code IS NULL OR newbie IS NULL
           OR channel IS NULL OR segment IS NULL OR visit IS NULL
           OR conversion IS NULL OR spend IS NULL
    ) AS rows_with_missing,
    count(*) FILTER (WHERE arm IS NULL) AS unmapped_arm,
    count(*) - count(DISTINCT customer_id) AS duplicate_ids,
    count(*) FILTER (WHERE spend < 0 OR history <= 0 OR recency < 1) AS invalid_amounts,
    -- Outcome logic: spend > 0 <=> converted, and converted => visited.
    count(*) FILTER (WHERE spend > 0 AND conversion = 0) AS spend_without_conversion,
    count(*) FILTER (WHERE spend = 0 AND conversion = 1) AS conversion_without_spend,
    count(*) FILTER (WHERE conversion = 1 AND visit = 0) AS conversion_without_visit,
    -- Population is past-year purchasers, so each must have bought mens or womens.
    count(*) FILTER (WHERE mens = 0 AND womens = 0) AS no_purchase_history
FROM customers;

```

In [6]:
quality = run_sql_file(con, "04_data_quality.sql")
quality.T.rename(columns={0: "count"})

,count
n_rows,64000
rows_with_missing,0
unmapped_arm,0
duplicate_ids,0
invalid_amounts,0
spend_without_conversion,0
conversion_without_spend,0
conversion_without_visit,0
no_purchase_history,0


**Result:** 64,000 rows and zero violations. The file matches its documentation, so no rows are dropped or imputed.

## 3. Per-arm summary

Group sizes and raw outcome rates for each arm. These are **descriptive** numbers: confidence intervals, tests and the multiple-comparison correction come in notebook 03.

Two spend metrics are kept apart on purpose:

- `spend_per_customer` averages over everyone, zeros included. It is the metric the experiment's randomization protects, so differences between arms estimate the email's causal effect on spend.
- `spend_per_converter` averages only over customers who bought. Whether someone buys can itself be changed by the email, so the converters in each arm are not comparable groups. Differences in this column are **not** causal effects on basket size.

In [7]:
show_sql("02_arm_summary.sql")

**`sql/02_arm_summary.sql`**
```sql
-- Per-arm summary of the experiment: group sizes and outcome rates.
-- Requires the `customers` table from 01_load_clean.sql.
--
-- spend_per_customer is the main spend metric (all customers, zeros included).
-- spend_per_converter / median_spend_converter are descriptive only: they condition
-- on conversion, which the email itself can change, so differences between arms
-- are not causal effects on basket size.

SELECT
    arm,
    count(*) AS n,
    count(*) / sum(count(*)) OVER () AS share,
    sum(visit) AS visits,
    avg(visit) AS visit_rate,
    sum(conversion) AS conversions,
    avg(conversion) AS conversion_rate,
    sum(spend) AS total_spend,
    avg(spend) AS spend_per_customer,
    stddev_samp(spend) AS spend_sd,
    avg(spend) FILTER (WHERE conversion = 1) AS spend_per_converter,
    median(spend) FILTER (WHERE conversion = 1) AS median_spend_converter
FROM customers
GROUP BY arm
ORDER BY arm;

```

In [8]:
arm_summary = run_sql_file(con, "02_arm_summary.sql").set_index("arm")
arm_summary.T

arm,control,mens,womens
n,"21,306.0000","21,307.0000","21,387.0000"
share,0.3329,0.3329,0.3342
visits,"2,262.0000","3,894.0000","3,238.0000"
visit_rate,0.1062,0.1828,0.1514
conversions,122.0000,267.0000,189.0000
conversion_rate,0.0057,0.0125,0.0088
total_spend,"13,908.3300","30,311.6900","23,038.1100"
spend_per_customer,0.6528,1.4226,1.0772
spend_sd,11.5882,17.7542,15.1161
spend_per_converter,114.0027,113.5269,121.8948


What the table shows, before any formal test:

- The three arms are almost exactly one third each (notebook 02 tests this formally).
- Both emails have higher visit, conversion and spend-per-customer rates than the control, and the Mens email is ahead of the Womens email on all three.
- Conversions are rare: about 1% of customers, roughly 120-270 converters per arm. Spend and conversion estimates will therefore be much noisier than visit estimates.
- Average spend among converters is similar across arms (about \$114-122). This hints that the emails raise spend mainly by turning more customers into buyers, but because it conditions on conversion, it is a hint, not a finding.

## 4. Per-segment summary

The same metrics by arm within each pre-treatment segment. The query uses `UNPIVOT` to turn the five segment columns into `(dimension, level)` pairs, so one `GROUP BY` covers all of them.

In [9]:
show_sql("03_segment_summary.sql")

**`sql/03_segment_summary.sql`**
```sql
-- Per-arm outcome summary within each level of the pre-treatment segment variables.
-- Requires the `customers` table from 01_load_clean.sql.
-- Output is long format: one row per (dimension, level, arm).
--
-- UNPIVOT turns the segment columns into (dimension, level) pairs so that a single
-- GROUP BY covers every dimension. Adding a dimension means adding one name to
-- both lists below.

WITH segments AS (
    SELECT
        arm,
        visit,
        conversion,
        spend,
        history_segment,
        channel,
        zip_code,
        CASE newbie WHEN 1 THEN 'new' ELSE 'existing' END AS newbie,
        purchase_group
    FROM customers
),
long AS (
    UNPIVOT segments
    ON history_segment, channel, zip_code, newbie, purchase_group
    INTO NAME dimension VALUE level
)
SELECT
    dimension,
    level,
    arm,
    count(*) AS n,
    avg(visit) AS visit_rate,
    sum(conversion) AS conversions,
    avg(conversion) AS conversion_rate,
    avg(spend) AS spend_per_customer
FROM long
GROUP BY dimension, level, arm
ORDER BY dimension, level, arm;

```

In [10]:
segment_summary = run_sql_file(con, "03_segment_summary.sql")
segment_summary.pivot_table(
    index=["dimension", "level"], columns="arm", values=["n", "conversions"], aggfunc="sum"
).astype(int)

conversions                   n              
arm                                  control mens womens control   mens womens
dimension       level                                                         
channel         Multichannel              18   44     36    2606   2577   2579
                Phone                     50  100     67    9327   9240   9454
                Web                       54  123     86    9373   9490   9354
history_segment 1) $0 - $100              32   71     61    7612   7724   7634
                2) $100 - $200            20   42     37    4836   4691   4727
                3) $200 - $350            28   63     26    4044   4090   4155
                4) $350 - $500            24   42     20    2124   2097   2188
                5) $500 - $750             9   22     26    1652   1597   1662
                6) $750 - $1,000           3   16      8     622    644    593
                7) $1,000 +                6   11     11     416    464    428
newbie          existing                  82  148     92   10611  10621  10624
                new                       40  119     97   10695  10686  10763
purchase_group  both                      20   53     31    2149   2181   2118
                mens_only                 53  110     59    9638   9558   9622
                womens_only               49  104     99    9519   9568   9647
zip_code        Rural                     26   44     37    3139   3243   3181
                Suburban                  48  118     73    9625   9501   9650
                Urban                     48  105     79    8542   8563   8556

The cell counts matter for what later notebooks can claim:

- Every segment has hundreds to thousands of customers per arm, so **visit rates** can be compared within segments.
- The high-spend history segments (`6) $750 - $1,000` and `7) $1,000 +`) have only a handful of conversions per arm (single digits to the mid-teens). Segment-level conversion and spend effects there will have very wide intervals, and notebook 06 treats them accordingly.

In [11]:
con.close()

## Takeaways

- The raw data is clean: no missing values, no logical inconsistencies, one spelling fix.
- The arms are balanced in size, and raw rates favour both emails, the Mens email most.
- Conversions are rare (~1%), which is the main statistical constraint for every later step: visit-based results will be the most reliable, spend-based results the noisiest.